# Produce functional area aggregations

This notebook produces aggregations for a given NHP Demand and Capacity model scenario. Please note the following ESSENTIAL requirements for running this notebook:

- The scenario must have been run with full model results. Provide the path to the full model results in the notebook widget at the top of the notebook, in the format `full-model-results/vx.x/PROVIDER/SCENARIO_NAME/SCENARIO_RUNTIME/`
- A .env file with the correct environment variables
- Run the `generate_token.ps1` file in your local machine Terminal _before_ running this notebook, which generates SAS tokens valid for 24 hours and sets them as Databricks secrets

## Setup

In [0]:
%cd ..
%pip install .

from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient

spark = DatabricksSession.builder.getOrCreate()
w = WorkspaceClient()
dbutils = w.dbutils
dbutils.library.restartPython()

In [0]:
from nhp.capacity.functional_areas.loading_helpers import (
    load_env_vars,
    extract_model_run_details,
    load_model_data,
    validate_result_path,
    load_default_results,
)
from nhp.capacity.functional_areas.aae import (
    process_aae,
    process_sdec_converted,
    qa_aae_results,
)

from nhp.capacity.functional_areas.op import (
    process_op,
    process_op_converted,
    qa_op_results,
)
from nhp.capacity.functional_areas.ip_daycase import (
    process_ip_daycase,
    qa_ip_daycase_results,
)
from nhp.capacity.functional_areas.ip_wards import (
    process_ip_wards,
    qa_ip_wards_results,
)
from nhp.capacity.functional_areas.saving_helpers import (
    upload_data,
    add_metadata_to_ats,
)
from nhp.capacity.functional_areas.processing_helpers import (
    get_tretspef_lookup,
    add_tretspef_type,
)
from datetime import datetime
import uuid

dbutils.widgets.text("capacity_model_version", "dev", "Capacity Model version")
dbutils.widgets.text("path_to_full_model_results", "", "Path to full model results")
dbutils.widgets.text("sites_aae", "ALL", "AAE sites")
dbutils.widgets.text("sites_op", "ALL", "OP sites")
dbutils.widgets.text("sites_ip", "ALL", "IP sites")

mapping_runtime = datetime.now().strftime(format="%Y%m%d-%H%M%S")

In [0]:
path_to_full_model_results = dbutils.widgets.get("path_to_full_model_results")

db_path_to_full_model_results = str(validate_result_path(path_to_full_model_results))

env_vars = load_env_vars()

demand_model_version, fyear, provider, scenario_name, scenario_runtime = (
    extract_model_run_details(path_to_full_model_results)
)

## A&E

In [0]:
# Load data
sites_aae = dbutils.widgets.get("sites_aae").split(",")
aae_original = load_model_data(demand_model_version, "aae", fyear, provider, sites_aae)
aae_model_results = spark.read.parquet(db_path_to_full_model_results + "aae")
sdec_groupings_per_run = process_sdec_converted(db_path_to_full_model_results)

# Process data
final_aae_df = process_aae(aae_original, aae_model_results, sdec_groupings_per_run)

# QA processed data
default_aae_results = load_default_results(
    db_path_to_full_model_results, "aae", sites_aae
)
qa_aae_results(default_aae_results, final_aae_df, sites_aae)

## OP

In [0]:
# Load data
sites_op = dbutils.widgets.get("sites_op").split(",")
op_original = load_model_data(demand_model_version, "op", fyear, provider, sites_op)
op_model_results = spark.read.parquet(db_path_to_full_model_results + "op")
op_converted = process_op_converted(db_path_to_full_model_results)

# Process data
final_op_df = process_op(op_original, op_model_results, op_converted)

# QA processed data
default_op_results = load_default_results(db_path_to_full_model_results, "op", sites_op)
qa_op_results(default_op_results, final_op_df, sites_op)

## IP

In [ ]:
# Load data
sites_ip = dbutils.widgets.get("sites_ip").split(",")
ip_original = load_model_data(demand_model_version, "ip", fyear, provider, sites_ip)
ip_model_results = spark.read.parquet(db_path_to_full_model_results + "ip")
tretspef_lookup_df = get_tretspef_lookup(
    excel_path="/Volumes/nhp/reference/files/daycase_tretspef_mapping.xlsx"
)
ip_original_mapped = add_tretspef_type(ip_original, tretspef_lookup_df)

## IP Daycase

In [0]:
# Process data
final_ip_daycase_df = process_ip_daycase(ip_original_mapped, ip_model_results)

# QA processed data
default_ip_results = load_default_results(db_path_to_full_model_results, "ip", sites_ip)
qa_ip_daycase_results(default_ip_results, final_ip_daycase_df, sites_ip)

## IP Wards

In [ ]:
# Process data
final_ip_wards_df = process_ip_wards(ip_original_mapped, ip_model_results)

# QA processed data
default_ip_results = load_default_results(db_path_to_full_model_results, "ip", sites_ip)
qa_ip_wards_results(default_ip_results, final_ip_wards_df, sites_ip)

## Upload results

In [0]:
storage_guid = str(uuid.uuid4())
metadata = {
    "PartitionKey": dbutils.widgets.get("capacity_model_version"),
    "RowKey": storage_guid,
    "app_version": demand_model_version,
    "scenario_name": scenario_name,
    "scenario_runtime": scenario_runtime,
    "dataset": provider,
    "mapping_runtime": mapping_runtime,
    "path_to_full_results_dir": path_to_full_model_results,
    "sites_aae": ",".join(str(site) for site in sites_aae),
    "sites_op": ",".join(str(site) for site in sites_op),
    "sites_ip": ",".join(str(site) for site in sites_ip),
}

upload_data(env_vars, metadata, final_aae_df, "aae")
upload_data(env_vars, metadata, final_op_df, "op")
upload_data(env_vars, metadata, final_ip_daycase_df, "ip_daycase")

## Add details to Azure Table Storage

In [0]:
add_metadata_to_ats(env_vars, metadata)